<a href="https://colab.research.google.com/github/doundo/ICT_voice_detect/blob/voice_generate/ICT_voice_generate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install --force-reinstall --no-cache-dir torch torchaudio soundfile noisereduce --quiet
!pip install noisereduce
!pip install librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 210.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 123.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 175.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.1/106.1 kB 211.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
#무음 제거

import os
import librosa
import soundfile as sf
from tqdm import tqdm
import numpy as np

input_dir = '/content/drive/MyDrive/ICT_voice_detect_project/test'
output_dir = '/content/drive/MyDrive/ICT_voice_detect_project/test_silence_remove'
os.makedirs(output_dir, exist_ok=True)

sample_rate = 16000
top_db = 20  # 더 민감하게 하고 싶다면 25~30

for filename in tqdm(os.listdir(input_dir)):
    if filename.endswith('.wav'):
        path = os.path.join(input_dir, filename)
        y, sr = librosa.load(path, sr=sample_rate)

        # 무음 제거
        intervals = librosa.effects.split(y, top_db=top_db)
        y_voiced = np.concatenate([y[start:end] for start, end in intervals]) if len(intervals) > 0 else np.array([])

        # 너무 짧은 경우는 저장 생략
        if len(y_voiced) < sample_rate * 0.3:
            print(f"⚠️ 저장 생략: {filename} (유효 길이 {len(y_voiced)/sample_rate:.2f}s)")
            continue

        out_path = os.path.join(output_dir, filename)
        sf.write(out_path, y_voiced, samplerate=sample_rate)


100%|██████████| 5632/5632 [04:22<00:00, 21.45it/s]


In [ ]:
#속도 0.5~2배속까지 조절된 음성 생성

import os
import torchaudio
import torchaudio.functional as F

test_folder = '/content/drive/MyDrive/ICT_voice_detect_project/my_voice_preprocess'
speed_factors = [round(x, 1) for x in list(
    set([0.1 * i for i in range(5, 21)]) - {1.0}
)]

for fname in os.listdir(test_folder):
    if not fname.endswith('.wav'):
        continue

    input_path = os.path.join(test_folder, fname)
    waveform, sample_rate = torchaudio.load(input_path)

    for factor in sorted(speed_factors):
        # 새로운 sample rate 계산
        new_sample_rate = int(sample_rate * factor)
        new_waveform = F.resample(waveform, sample_rate, new_sample_rate)

        # 다시 원래 sample rate로 리샘플링 (파일 저장을 위해)
        adjusted_waveform = F.resample(new_waveform, new_sample_rate, sample_rate)

        # 파일 저장
        name, ext = os.path.splitext(fname)
        new_fname = f"{name}_speed_{factor}.wav"
        output_path = os.path.join(test_folder, new_fname)
        torchaudio.save(output_path, adjusted_waveform, sample_rate)

print("✅ 속도 조절된 wav 파일 저장 완료.")


✅ 속도 조절된 wav 파일 저장 완료.


In [ ]:
#1초 분할

import os
import torchaudio
import torchaudio.transforms as T
from tqdm import tqdm
import soundfile as sf

# 입력/출력 경로
input_dir = '/content/drive/MyDrive/ICT_voice_detect_project/test_silence_remove'  # 무음 제거 & 증강된 파일
output_dir = '/content/drive/MyDrive/ICT_voice_detect_project/split_1sec_test'
os.makedirs(output_dir, exist_ok=True)

# 설정값
sample_rate = 16000
chunk_duration = 1.0  # 초
chunk_size = int(sample_rate * chunk_duration)

chunk_count = 0

for filename in tqdm(sorted(os.listdir(input_dir))):
    if filename.endswith(".wav"):
        filepath = os.path.join(input_dir, filename)
        waveform, sr = torchaudio.load(filepath)

        # 샘플레이트 맞추기
        if sr != sample_rate:
            resampler = T.Resample(orig_freq=sr, new_freq=sample_rate)
            waveform = resampler(waveform)

        total_samples = waveform.shape[1]
        num_chunks = total_samples // chunk_size

        for i in range(num_chunks):
            start = i * chunk_size
            end = start + chunk_size
            chunk = waveform[:, start:end]

            # 저장 경로 구성
            out_filename = f"{filename[:-4]}_chunk_{i:04d}.wav"
            out_path = os.path.join(output_dir, out_filename)

            # 저장 (soundfile 사용)
            sf.write(out_path, chunk.squeeze(0).numpy(), samplerate=sample_rate)
            chunk_count += 1

print(f"✅ 총 {chunk_count}개의 1초 단위 WAV 파일 저장 완료")


100%|██████████| 5632/5632 [03:54<00:00, 24.00it/s]

✅ 총 9397개의 1초 단위 WAV 파일 저장 완료


In [ ]:
#내 음성 train, val, test 분할하기

import os
import random
import shutil

source_dir = "/content/drive/MyDrive/ICT_voice_detect_project/split_1sec_my_voice_preprocess"
base_output_dir = "/content/drive/MyDrive/ICT_voice_detect_project/split_splited_1sec_dataset"

train_dir = os.path.join(base_output_dir, "train")
val_dir = os.path.join(base_output_dir, "val")
test_dir = os.path.join(base_output_dir, "test")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

all_files = [f for f in os.listdir(source_dir) if f.endswith(".wav")]
random.shuffle(all_files)

total = len(all_files)
train_end = int(total * 0.7)
val_end = train_end + int(total * 0.15)

train_files = all_files[:train_end]
val_files = all_files[train_end:val_end]
test_files = all_files[val_end:]

def copy_files(file_list, dest_dir):
    for f in file_list:
        shutil.copy2(os.path.join(source_dir, f), os.path.join(dest_dir, f))

copy_files(train_files, train_dir)
copy_files(val_files, val_dir)
copy_files(test_files, test_dir)

len(train_files), len(val_files), len(test_files)


KeyboardInterrupt: 

In [ ]:
#다른 사람 음성 train, val, test 분할

import os
import shutil
import random

# 원본 경로
source_dirs = "/content/drive/MyDrive/ICT_voice_detect_project/split_1sec_test"

# 목표 샘플 수
target_sample_count = 7517
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# 출력 경로
output_base = "/content/drive/MyDrive/ICT_voice_detect_project/final_dataset"
output_dirs = {
    "train": os.path.join(output_base, "train_other"),
    "val": os.path.join(output_base, "val_other"),
    "test": os.path.join(output_base, "test_other")
}

# 디렉토리 생성 (이미 있으면 넘어감)
for path in output_dirs.values():
    os.makedirs(path, exist_ok=True)

# .wav 파일 수집
all_files = []
for fname in os.listdir(source_dirs):
    if fname.lower().endswith(".wav"):
        full_path = os.path.join(source_dirs, fname)
        all_files.append(full_path)

# 무작위 셔플 및 선택
random.seed(42)
random.shuffle(all_files)
selected_files = all_files[:target_sample_count]

# 비율 분할
n_train = int(target_sample_count * train_ratio)
n_val = int(target_sample_count * val_ratio)

splits = {
    "train": selected_files[:n_train],
    "val": selected_files[n_train:n_train + n_val],
    "test": selected_files[n_train + n_val:]
}

# 복사 실행
for split_name, file_list in splits.items():
    for src_path in file_list:
        fname = os.path.basename(src_path)
        dst_path = os.path.join(output_dirs[split_name], fname)
        shutil.copy2(src_path, dst_path)

print("✅ 복사 완료:")
print(f" - Train: {len(splits['train'])}개")
print(f" - Val: {len(splits['val'])}개")
print(f" - Test: {len(splits['test'])}개")


✅ 복사 완료:
 - Train: 5261개
 - Val: 1127개
 - Test: 1129개


In [ ]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
import pandas as pd
from tqdm import tqdm

# 설정
sample_rate = 16000
n_mfcc = 13
mfcc_transform = T.MFCC(
    sample_rate=sample_rate,
    n_mfcc=n_mfcc,
    melkwargs={"n_fft": 400, "hop_length": 160, "n_mels": 40}
)

# 경로
base_dir = "/content/drive/MyDrive/ICT_voice_detect_project/final_dataset"
folders = {
    "train_my_voice": 1,
    "train_other": 0,
    "val_my_voice": 1,
    "val_other": 0,
    "test_my_voice": 1,
    "test_other": 0
}

output_csv_path = "/content/drive/MyDrive/ICT_voice_detect_project/voice_mfcc_dataset.csv"

# CSV 헤더 생성
columns = ["file_name", "label"] + [f"mfcc_{i+1}" for i in range(n_mfcc)]
with open(output_csv_path, "w") as f:
    f.write(",".join(columns) + "\n")

# 처리 및 저장
for folder_name, label in folders.items():
    folder_path = os.path.join(base_dir, folder_name)
    for fname in tqdm(os.listdir(folder_path), desc=folder_name):
        if fname.lower().endswith(".wav"):
            fpath = os.path.join(folder_path, fname)
            try:
                waveform, sr = torchaudio.load(fpath)
                if sr != sample_rate:
                    resampler = T.Resample(orig_freq=sr, new_freq=sample_rate)
                    waveform = resampler(waveform)
                mfcc = mfcc_transform(waveform).squeeze(0).transpose(0, 1)
                mfcc_mean = mfcc.mean(dim=0).numpy()
                values = [fname, str(label)] + [str(v) for v in mfcc_mean]
                with open(output_csv_path, "a") as f:
                    f.write(",".join(values) + "\n")
            except Exception as e:
                print(f"⚠️ 오류 발생: {fname} - {str(e)}")


test_other: 100%|██████████| 1129/1129 [00:20<00:00, 54.30it/s]
